# Prepare Classification Dataset

This notebook creates cropped dice images for the classifier
from the annotated detection dataset.

Goals:
- load train and validation splits
- crop individual dice from bounding boxes
- optionally exclude problematic source images
- save crops into class-specific folders

In [ ]:
from pathlib import Path
import json
import shutil
import random

import cv2
import numpy as np

from project_config import (
    PROJECT_ROOT,
    TRAIN_JSON,
    VAL_JSON,
    CLASSIFICATION_DIR,
    SEED,
)
from utils.classification_data import (
    resolve_image_path,
    jitter_box_xyxy,
    crop_box,
)

In [ ]:
random.seed(SEED)
np.random.seed(SEED)

TRAIN_OUT = CLASSIFICATION_DIR / "train"
VAL_OUT = CLASSIFICATION_DIR / "val"
EXCLUDED_OUT = CLASSIFICATION_DIR / "excluded_number_dice"

CLASS_NAMES = [1, 2, 3, 4, 5, 6]

BASE_MARGIN = 0.12
TRAIN_BOX_JITTER = 0.08
VAL_BOX_JITTER = 0.00

RECREATE_OUTPUT_DIRS = True
SAVE_EXCLUDED_CROPS = True

MANUAL_EXCLUDE_LIST = {
    "i.rf.89b3074c7102e958651696ff0ad59deb_4_18.jpg",
    "i.rf.89b3074c7102e958651696ff0ad59deb_4_20.jpg",
    "i.rf.89b3074c7102e958651696ff0ad59deb_4_31.jpg",
    "i.rf.254d1630921ad675efaa154e08a5e098_187_2.jpg",
}

In [ ]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_JSON exists:", TRAIN_JSON.exists())
print("VAL_JSON exists:", VAL_JSON.exists())

assert TRAIN_JSON.exists(), f"Missing: {TRAIN_JSON}"
assert VAL_JSON.exists(), f"Missing: {VAL_JSON}"

In [ ]:
with open(TRAIN_JSON, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(VAL_JSON, "r", encoding="utf-8") as f:
    val_data = json.load(f)

print("Train samples:", len(train_data))
print("Val samples:", len(val_data))

In [ ]:
if RECREATE_OUTPUT_DIRS and CLASSIFICATION_DIR.exists():
    shutil.rmtree(CLASSIFICATION_DIR)

for split_dir in [TRAIN_OUT, VAL_OUT]:
    for cls in CLASS_NAMES:
        (split_dir / str(cls)).mkdir(parents=True, exist_ok=True)

if SAVE_EXCLUDED_CROPS:
    EXCLUDED_OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
def should_exclude_sample(sample):
    image_name = Path(sample["image_path"]).name
    return image_name in MANUAL_EXCLUDE_LIST


def save_excluded_crop(crop, image_path, sample_idx, crop_idx, label, split_name):
    if not SAVE_EXCLUDED_CROPS:
        return

    out_name = f"{split_name}__label{label}__{Path(image_path).stem}_{sample_idx}_{crop_idx}.jpg"
    out_path = EXCLUDED_OUT / out_name
    cv2.imwrite(str(out_path), crop)

In [ ]:
def process_split(data, out_dir, jitter_ratio=0.0, split_name="train"):
    saved = 0
    skipped = 0
    excluded = 0

    for sample_idx, sample in enumerate(data):
        image_path = resolve_image_path(sample["image_path"], PROJECT_ROOT)

        boxes = np.asarray(sample["boxes"], dtype=np.float32).reshape(-1, 4)
        class_ids = np.asarray(sample["class_ids"], dtype=np.int32).reshape(-1)

        if len(boxes) != len(class_ids):
            skipped += 1
            continue

        image = cv2.imread(str(image_path))
        if image is None:
            skipped += 1
            continue

        exclude_whole_sample = should_exclude_sample(sample)

        for i, (box, cls_id) in enumerate(zip(boxes, class_ids)):
            label = int(cls_id) + 1
            if label not in CLASS_NAMES:
                skipped += 1
                continue

            box_used = jitter_box_xyxy(box, jitter_ratio=jitter_ratio)
            if box_used is None:
                skipped += 1
                continue

            crop = crop_box(image, box_used, margin=BASE_MARGIN)
            if crop is None:
                skipped += 1
                continue

            if exclude_whole_sample:
                save_excluded_crop(crop, image_path, sample_idx, i, label, split_name)
                excluded += 1
                continue

            save_name = f"{Path(image_path).stem}_{sample_idx}_{i}.jpg"
            save_path = out_dir / str(label) / save_name

            ok = cv2.imwrite(str(save_path), crop)
            if ok:
                saved += 1
            else:
                skipped += 1

    print(f"{split_name}: saved={saved}, excluded={excluded}, skipped={skipped}")
    return saved, excluded, skipped

In [ ]:
train_saved, train_excluded, train_skipped = process_split(
    train_data,
    TRAIN_OUT,
    jitter_ratio=TRAIN_BOX_JITTER,
    split_name="train",
)

val_saved, val_excluded, val_skipped = process_split(
    val_data,
    VAL_OUT,
    jitter_ratio=VAL_BOX_JITTER,
    split_name="val",
)

print("\nDONE")
print("TRAIN_OUT:", TRAIN_OUT)
print("VAL_OUT:", VAL_OUT)
print("EXCLUDED_OUT:", EXCLUDED_OUT if SAVE_EXCLUDED_CROPS else "disabled")